# IL&FS Crisis — Derivatives Data Cleaning
### `01_data_cleaning.ipynb`

This notebook performs **data cleaning only** for the study of the 2018 IL&FS
liquidity crisis and its impact on the Indian derivatives market (Nifty and
Bank Nifty).

No pricing, Greeks, hedging, event-study, or statistical analysis is
performed here — this notebook's sole job is to turn the raw NSE files in
`Data/Raw/` into clean, analysis-ready CSVs in `Data/Cleaned/`.

**Run top to bottom.** Only `BASE_DIR` below needs to be adjusted if your
folder location differs.

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================
# Adjust BASE_DIR if your project folder is located elsewhere.
BASE_DIR = Path(r"C:\Users\sanka\OneDrive\Documents\Derivatives")

RAW_DIR = BASE_DIR / "Data" / "Raw"
CLEANED_DIR = BASE_DIR / "Data" / "Cleaned"

# Raw files are read-only inputs; never overwritten
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory     : {RAW_DIR}")
print(f"Cleaned data directory : {CLEANED_DIR}")


Raw data directory     : C:\Users\sanka\OneDrive\Documents\Derivatives\Data\Raw
Cleaned data directory : C:\Users\sanka\OneDrive\Documents\Derivatives\Data\Cleaned


## Utility: Cleaning Report

In [3]:
# ============================================================
# UTILITY FUNCTION: STANDARD CLEANING REPORT
# ============================================================
def print_cleaning_report(name: str, rows_before: int, rows_after: int,
                           duplicates_removed: int, missing_values: pd.Series) -> None:
    """Print a consistent before/after cleaning summary for a dataset."""
    print("-" * 60)
    print(f"Cleaning Report: {name}")
    print("-" * 60)
    print(f"Rows before cleaning : {rows_before}")
    print(f"Rows after cleaning  : {rows_after}")
    print(f"Duplicates removed   : {duplicates_removed}")
    print("Missing values per column:")
    print(missing_values)
    print()


## 1. Spot Data
Files: `nifty_spot.csv`, `banknifty_spot.csv`

In [4]:
# ============================================================
# SPOT DATA CLEANING FUNCTION
# ============================================================
def clean_spot_data(file_name: str, label: str) -> pd.DataFrame:
    """Clean a Nifty/Bank Nifty spot index file."""
    df = pd.read_csv(RAW_DIR / file_name)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    rows_before = len(df)

    # Rename columns to standardised names
    df = df.rename(columns={
        "Shares Traded": "SharesTraded",
        "Turnover (₹ Cr)": "Turnover",
    })

    # Parse and standardise the Date column
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

    # Convert numeric columns safely
    numeric_cols = ["Open", "High", "Low", "Close", "SharesTraded", "Turnover"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove exact duplicate rows
    duplicates_removed = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)

    # Sort chronologically
    df = df.sort_values("Date").reset_index(drop=True)

    # Check missing values
    missing_values = df.isna().sum()

    # Save dates in YYYY-MM-DD format
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    print_cleaning_report(label, rows_before, len(df), duplicates_removed, missing_values)

    return df


nifty_spot_clean = clean_spot_data("nifty_spot.csv", "Nifty Spot")
nifty_spot_clean.to_csv(CLEANED_DIR / "nifty_spot_clean.csv", index=False)
nifty_spot_clean.head()


------------------------------------------------------------
Cleaning Report: Nifty Spot
------------------------------------------------------------
Rows before cleaning : 143
Rows after cleaning  : 143
Duplicates removed   : 0
Missing values per column:
Date            0
Open            0
High            0
Low             0
Close           0
SharesTraded    0
Turnover        0
dtype: int64



C:\Users\sanka\AppData\Local\Temp\ipykernel_67680\2624364798.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")


,Date,Open,High,Low,Close,SharesTraded,Turnover
0,2018-06-01,10738.45,10764.75,10681.50,10696.20,227355567,12965.01
1,2018-06-04,10765.95,10770.30,10618.35,10628.50,210148313,14120.49
2,2018-06-05,10630.70,10633.15,10550.90,10593.15,172898027,10027.54
3,2018-06-06,10603.45,10698.35,10587.50,10684.65,195761429,9920.75
4,2018-06-07,10722.60,10818.00,10722.60,10768.35,227663151,11315.24


In [5]:
banknifty_spot_clean = clean_spot_data("banknifty_spot.csv", "Bank Nifty Spot")
banknifty_spot_clean.to_csv(CLEANED_DIR / "banknifty_spot_clean.csv", index=False)
banknifty_spot_clean.head()


------------------------------------------------------------
Cleaning Report: Bank Nifty Spot
------------------------------------------------------------
Rows before cleaning : 143
Rows after cleaning  : 143
Duplicates removed   : 0
Missing values per column:
Date            0
Open            0
High            0
Low             0
Close           0
SharesTraded    0
Turnover        0
dtype: int64



C:\Users\sanka\AppData\Local\Temp\ipykernel_67680\2624364798.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")


,Date,Open,High,Low,Close,SharesTraded,Turnover
0,2018-06-01,26913.40,26996.80,26658.55,26692.80,132336254,4017.91
1,2018-06-04,27023.70,27047.55,26205.90,26257.55,137561064,7188.74
2,2018-06-05,26263.00,26342.40,26069.55,26251.00,107940100,3169.87
3,2018-06-06,26266.90,26425.20,26147.80,26367.60,144159415,3007.56
4,2018-06-07,26577.95,26766.95,26483.90,26517.80,131294980,3421.62


## 2. Futures Data
Files: `nifty_futures.csv`, `banknifty_futures.csv`

Raw NSE F&O bhavcopy-style files are assumed to contain an `OPTION_TYPE`
column where futures contracts are marked `XX` and any `FF` rows (not
relevant to this dataset) are excluded. For every trading day, only the
**nearest-expiry** contract is retained.

In [ ]:
# ============================================================
# FUTURES DATA CLEANING FUNCTION
# ============================================================
def clean_futures_data(file_name: str, label: str) -> pd.DataFrame:
    """Clean a Nifty/Bank Nifty futures file, retaining all futures expiries
for each trading day."""
    df = pd.read_csv(RAW_DIR / file_name)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    rows_before = len(df)

    # Remove rows tagged "FF" and keep only rows tagged "XX" (futures contracts)
    df = df[df["OPTION_TYPE"] != "FF"]
    df = df[df["OPTION_TYPE"] == "XX"].copy()

    # Convert TIMESTAMP -> Date, EXPIRY_DT -> Expiry
    df["Date"] = pd.to_datetime(df["TIMESTAMP"], dayfirst=True, errors="coerce")
    df["Expiry"] = pd.to_datetime(df["EXPIRY_DT"], dayfirst=True, errors="coerce")

    # Convert numeric columns safely
    numeric_source_cols = [
        "OPENING_PRICE", "TRADE_HIGH_PRICE", "TRADE_LOW_PRICE", "CLOSING_PRICE",
        "SETTLE_PRICE", "TOT_TRADED_QTY", "TOT_TRADED_VAL", "OPEN_INT",
        "CHANGE_IN_OI", "UNDERLYING_VALUE",
    ]
    for col in numeric_source_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Rename columns to standardised names
    df = df.rename(columns={
        "OPENING_PRICE": "Open",
        "TRADE_HIGH_PRICE": "High",
        "TRADE_LOW_PRICE": "Low",
        "CLOSING_PRICE": "Close",
        "SETTLE_PRICE": "SettlementPrice",
        "TOT_TRADED_QTY": "Volume",
        "TOT_TRADED_VAL": "Turnover",
        "OPEN_INT": "OpenInterest",
        "CHANGE_IN_OI": "ChangeInOI",
        "UNDERLYING_VALUE": "Underlying",
    })

    # Sort by Date and Expiry
    df = df.sort_values(["Date", "Expiry"]).reset_index(drop=True)

    # Keep only the required, consistently-named columns
    final_cols = [
        "Date", "Expiry", "Open", "High", "Low", "Close", "SettlementPrice",
        "Volume", "Turnover", "OpenInterest", "ChangeInOI", "Underlying",
    ]
    df = df[final_cols].copy()

    # Remove exact duplicate rows
    duplicates_removed = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)

    # Sort by Date
    df = df.sort_values("Date").reset_index(drop=True)

    # Check missing values
    missing_values = df.isna().sum()

    # Save dates in YYYY-MM-DD format
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
    df["Expiry"] = df["Expiry"].dt.strftime("%Y-%m-%d")

    print_cleaning_report(label, rows_before, len(df), duplicates_removed, missing_values)

    return df


nifty_futures_clean = clean_futures_data("nifty_futures.csv", "Nifty Futures")
nifty_futures_clean.to_csv(CLEANED_DIR / "nifty_futures_clean.csv", index=False)
nifty_futures_clean.head()


------------------------------------------------------------
Cleaning Report: Nifty Futures
------------------------------------------------------------
Rows before cleaning : 436
Rows after cleaning  : 143
Duplicates removed   : 0
Missing values per column:
Date                0
Expiry              0
Open                0
High                0
Low                 0
Close               0
SettlementPrice     0
Volume              0
Turnover            0
OpenInterest        0
ChangeInOI          0
Underlying         68
dtype: int64



,Date,Expiry,Open,High,Low,Close,SettlementPrice,Volume,Turnover,OpenInterest,ChangeInOI,Underlying
0,2018-06-01,2018-06-28,10725.00,10744.00,10669.95,10683.90,10683.90,8243775.0,882711.68,19495050.0,501075.0,10696.20
1,2018-06-04,2018-06-28,10732.20,10732.20,10587.65,10598.70,10598.70,9649575.0,1027095.11,20347275.0,852225.0,NaN
2,2018-06-05,2018-06-28,10598.70,10630.00,10552.00,10596.45,10596.45,8610075.0,911654.36,20465400.0,118125.0,NaN
3,2018-06-06,2018-06-28,10604.75,10701.00,10575.00,10686.25,10686.25,10079925.0,1072932.54,20637825.0,172425.0,10684.65
4,2018-06-07,2018-06-28,10725.25,10814.05,10725.25,10765.20,10765.20,8984775.0,967916.42,21831825.0,1194000.0,NaN


In [7]:
banknifty_futures_clean = clean_futures_data("banknifty_futures.csv", "Bank Nifty Futures")
banknifty_futures_clean.to_csv(CLEANED_DIR / "banknifty_futures_clean.csv", index=False)
banknifty_futures_clean.head()


------------------------------------------------------------
Cleaning Report: Bank Nifty Futures
------------------------------------------------------------
Rows before cleaning : 436
Rows after cleaning  : 143
Duplicates removed   : 0
Missing values per column:
Date                0
Expiry              0
Open                0
High                0
Low                 0
Close               0
SettlementPrice     0
Volume              0
Turnover            0
OpenInterest        0
ChangeInOI          0
Underlying         68
dtype: int64



C:\Users\sanka\AppData\Local\Temp\ipykernel_67680\1925800335.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["TIMESTAMP"], dayfirst=True, errors="coerce")
C:\Users\sanka\AppData\Local\Temp\ipykernel_67680\1925800335.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Expiry"] = pd.to_datetime(df["EXPIRY_DT"], dayfirst=True, errors="coerce")


,Date,Expiry,Open,High,Low,Close,SettlementPrice,Volume,Turnover,OpenInterest,ChangeInOI,Underlying
0,2018-06-01,2018-06-28,26726.00,26887.0,26620.00,26669.25,26669.25,4104680.0,1097098.86,2711760.0,-128520.0,26692.8
1,2018-06-04,2018-06-28,26890.00,26899.0,26136.10,26190.10,26190.10,4811960.0,1270713.28,2608360.0,-103400.0,NaN
2,2018-06-05,2018-06-28,26140.00,26329.0,26079.05,26257.45,26257.45,4006640.0,1050309.04,2480520.0,-127840.0,NaN
3,2018-06-06,2018-06-28,26277.95,26450.0,26133.55,26391.50,26391.50,4745560.0,1248113.56,2521160.0,40640.0,26367.6
4,2018-06-07,2018-06-28,26563.00,26798.0,26505.00,26567.35,26567.35,5202840.0,1386913.19,2600920.0,79760.0,NaN


## 3. Options Data
Files: `nifty_options.csv`, `banknifty_options.csv`

The **complete option chain** is preserved — no strikes, expiries, or
zero-volume/illiquid contracts are removed. Only exact duplicate rows are
dropped.

In [8]:
# ============================================================
# OPTIONS DATA CLEANING FUNCTION
# ============================================================
def clean_options_data(file_name: str, label: str) -> pd.DataFrame:
    """Clean a Nifty/Bank Nifty options file, preserving the complete chain."""
    df = pd.read_csv(RAW_DIR / file_name)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    rows_before = len(df)

    # Convert date columns
    df["Date"] = pd.to_datetime(df["TIMESTAMP"], dayfirst=True, errors="coerce")
    df["Expiry"] = pd.to_datetime(df["EXPIRY_DT"], dayfirst=True, errors="coerce")

    # Convert numeric columns safely
    numeric_source_cols = [
        "STRIKE_PRICE", "OPENING_PRICE", "TRADE_HIGH_PRICE", "TRADE_LOW_PRICE",
        "CLOSING_PRICE", "LAST_TRADED_PRICE", "PREV_CLS", "SETTLE_PRICE",
        "TOT_TRADED_QTY", "TOT_TRADED_VAL", "OPEN_INT", "CHANGE_IN_OI",
        "MARKET_LOT", "UNDERLYING_VALUE",
    ]
    for col in numeric_source_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Rename columns to standardised names
    df = df.rename(columns={
        "STRIKE_PRICE": "Strike",
        "OPTION_TYPE": "OptionType",
        "OPENING_PRICE": "Open",
        "TRADE_HIGH_PRICE": "High",
        "TRADE_LOW_PRICE": "Low",
        "CLOSING_PRICE": "Close",
        "SETTLE_PRICE": "SettlementPrice",
        "TOT_TRADED_QTY": "Volume",
        "TOT_TRADED_VAL": "Turnover",
        "OPEN_INT": "OpenInterest",
        "CHANGE_IN_OI": "ChangeInOI",
        "UNDERLYING_VALUE": "Underlying",
    })

    # Keep the complete option chain: no filtering of strikes, expiries,
    # zero-volume, illiquid, OTM/ITM contracts — remove exact duplicates only
    duplicates_removed = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)

    # Sort by Date, Expiry, Strike, OptionType
    df = df.sort_values(["Date", "Expiry", "Strike", "OptionType"]).reset_index(drop=True)

    # Check missing values
    missing_values = df.isna().sum()

    # Save dates in YYYY-MM-DD format
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
    df["Expiry"] = df["Expiry"].dt.strftime("%Y-%m-%d")

    print_cleaning_report(label, rows_before, len(df), duplicates_removed, missing_values)

    return df


nifty_options_clean = clean_options_data("nifty_options.csv", "Nifty Options")
nifty_options_clean.to_csv(CLEANED_DIR / "nifty_options_clean.csv", index=False)
nifty_options_clean.head()


------------------------------------------------------------
Cleaning Report: Nifty Options
------------------------------------------------------------
Rows before cleaning : 331944
Rows after cleaning  : 331944
Duplicates removed   : 0
Missing values per column:
TIMESTAMP                 0
INSTRUMENT                0
SYMBOL                    0
EXPIRY_DT                 0
Strike                    0
OptionType                0
MARKET_TYPE            1082
Open                   1082
High                   1082
Low                    1082
Close                  1082
LAST_TRADED_PRICE      1082
PREV_CLS               1082
SettlementPrice        1082
Volume                 1082
Turnover               1082
OpenInterest           1082
ChangeInOI             1082
MARKET_LOT             1082
Underlying           158458
Date                      0
Expiry                    0
dtype: int64



,TIMESTAMP,INSTRUMENT,SYMBOL,EXPIRY_DT,Strike,OptionType,MARKET_TYPE,Open,High,Low,...,PREV_CLS,SettlementPrice,Volume,Turnover,OpenInterest,ChangeInOI,MARKET_LOT,Underlying,Date,Expiry
0,01-Jun-2018,OPTIDX,NIFTY,28-Jun-2018,3500,CE,N,7195.0,7195.0,7117.95,...,7150.45,7119.8,1050.0,111.61,81600.0,-150.0,75.0,10696.2,2018-06-01,2018-06-28
1,01-Jun-2018,OPTIDX,NIFTY,28-Jun-2018,3500,PE,N,1.4,1.4,1.00,...,1.15,1.1,450.0,15.76,15150.0,150.0,75.0,10696.2,2018-06-01,2018-06-28
2,01-Jun-2018,OPTIDX,NIFTY,28-Jun-2018,3600,CE,N,0.0,0.0,0.00,...,3653.75,7114.2,0.0,0.00,0.0,0.0,75.0,10696.2,2018-06-01,2018-06-28
3,01-Jun-2018,OPTIDX,NIFTY,28-Jun-2018,3600,PE,N,0.0,0.0,0.00,...,32.45,0.0,0.0,0.00,0.0,0.0,75.0,10696.2,2018-06-01,2018-06-28
4,01-Jun-2018,OPTIDX,NIFTY,28-Jun-2018,3700,CE,N,0.0,0.0,0.00,...,3592.35,7014.7,0.0,0.00,0.0,0.0,75.0,10696.2,2018-06-01,2018-06-28


In [9]:
banknifty_options_clean = clean_options_data("banknifty_options.csv", "Bank Nifty Options")
banknifty_options_clean.to_csv(CLEANED_DIR / "banknifty_options_clean.csv", index=False)
banknifty_options_clean.head()


------------------------------------------------------------
Cleaning Report: Bank Nifty Options
------------------------------------------------------------
Rows before cleaning : 141704
Rows after cleaning  : 141704
Duplicates removed   : 0
Missing values per column:
TIMESTAMP                0
INSTRUMENT               0
SYMBOL                   0
EXPIRY_DT                0
Strike                   0
OptionType               0
MARKET_TYPE           2420
Open                  2420
High                  2420
Low                   2420
Close                 2420
LAST_TRADED_PRICE     2420
PREV_CLS              2420
SettlementPrice       2420
Volume                2420
Turnover              2420
OpenInterest          2420
ChangeInOI            2420
MARKET_LOT            2420
Underlying           69682
Date                     0
Expiry                   0
dtype: int64



,TIMESTAMP,INSTRUMENT,SYMBOL,EXPIRY_DT,Strike,OptionType,MARKET_TYPE,Open,High,Low,...,PREV_CLS,SettlementPrice,Volume,Turnover,OpenInterest,ChangeInOI,MARKET_LOT,Underlying,Date,Expiry
0,01-Jun-2018,OPTIDX,BANKNIFTY,07-Jun-2018,23200,CE,N,0.0,0.0,0.0,...,2123.75,3518.6,0.0,0.00,0.0,0.0,40.0,26692.8,2018-06-01,2018-06-07
1,01-Jun-2018,OPTIDX,BANKNIFTY,07-Jun-2018,23200,PE,N,2.0,2.0,1.6,...,294.45,1.8,6440.0,1494.19,5680.0,5680.0,40.0,26692.8,2018-06-01,2018-06-07
2,01-Jun-2018,OPTIDX,BANKNIFTY,07-Jun-2018,23300,CE,N,0.0,0.0,0.0,...,2048.60,3418.7,0.0,0.00,0.0,0.0,40.0,26692.8,2018-06-01,2018-06-07
3,01-Jun-2018,OPTIDX,BANKNIFTY,07-Jun-2018,23300,PE,N,0.0,0.0,0.0,...,318.15,0.0,0.0,0.00,0.0,0.0,40.0,26692.8,2018-06-01,2018-06-07
4,01-Jun-2018,OPTIDX,BANKNIFTY,07-Jun-2018,23400,CE,N,0.0,0.0,0.0,...,1975.70,3318.8,0.0,0.00,0.0,0.0,40.0,26692.8,2018-06-01,2018-06-07


## 4. India VIX
File: `india_vix.csv`

In [10]:
# ============================================================
# INDIA VIX CLEANING
# ============================================================
def clean_vix_data(file_name: str, label: str) -> pd.DataFrame:
    """Clean the India VIX file, keeping only Date and VIX close level."""
    df = pd.read_csv(RAW_DIR / file_name)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    rows_before = len(df)

    # Parse Date
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

    # Convert Close to numeric before keeping it
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

    # Keep only Date and Close, dropping Adj Close, High, Low, Open, Volume
    df = df[["Date", "Close"]].copy()

    # Rename Close -> VIX
    df = df.rename(columns={"Close": "VIX"})

    # Remove exact duplicate rows
    duplicates_removed = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)

    # Sort by Date
    df = df.sort_values("Date").reset_index(drop=True)

    # Check missing values
    missing_values = df.isna().sum()

    # Save dates in YYYY-MM-DD format
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    print_cleaning_report(label, rows_before, len(df), duplicates_removed, missing_values)

    return df


india_vix_clean = clean_vix_data("india_vix.csv", "India VIX")
india_vix_clean.to_csv(CLEANED_DIR / "india_vix_clean.csv", index=False)
india_vix_clean.head()


------------------------------------------------------------
Cleaning Report: India VIX
------------------------------------------------------------
Rows before cleaning : 144
Rows after cleaning  : 144
Duplicates removed   : 0
Missing values per column:
Date    1
VIX     1
dtype: int64



,Date,VIX
0,2018-06-01,13.57
1,2018-06-04,13.88
2,2018-06-05,13.32
3,2018-06-06,12.55
4,2018-06-07,12.78


## 5. Risk-Free Rate
File: `risk_free_rate.xlsx`

In [11]:
# ============================================================
# RISK-FREE RATE CLEANING
# ============================================================
def clean_risk_free_rate(file_name: str, label: str) -> pd.DataFrame:
    """Clean the T-bill cut-off yield file. No forward-fill/interpolation
    here — that happens later during dataset merging."""
    df = pd.read_excel(RAW_DIR / file_name)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    rows_before = len(df)

    # Rename columns to standardised names
    df = df.rename(columns={
        "Date of Auction": "Date",
        "Implicit Yield at Cut-off Price (percent)": "RiskFreeRate",
    })

    # Parse Date
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

    # Convert RiskFreeRate to numeric, then percentage -> decimal (e.g. 8.12 -> 0.0812)
    df["RiskFreeRate"] = pd.to_numeric(df["RiskFreeRate"], errors="coerce") / 100

    # Remove exact duplicate rows
    duplicates_removed = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)

    # Sort by Date
    df = df.sort_values("Date").reset_index(drop=True)

    # Check missing values
    missing_values = df.isna().sum()

    # Save dates in YYYY-MM-DD format
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    print_cleaning_report(label, rows_before, len(df), duplicates_removed, missing_values)

    return df


risk_free_rate_clean = clean_risk_free_rate("risk_free_rate.xlsx", "Risk-Free Rate")
risk_free_rate_clean.to_csv(CLEANED_DIR / "risk_free_rate_clean.csv", index=False)
risk_free_rate_clean.head()


------------------------------------------------------------
Cleaning Report: Risk-Free Rate
------------------------------------------------------------
Rows before cleaning : 30
Rows after cleaning  : 30
Duplicates removed   : 0
Missing values per column:
Date            0
RiskFreeRate    0
dtype: int64



,Date,RiskFreeRate
0,2018-06-06,0.065219
1,2018-06-13,0.065219
2,2018-06-20,0.064805
3,2018-06-27,0.065219
4,2018-07-04,0.064805


## 6. Completion Summary

In [12]:
# ============================================================
# FINAL OUTPUT CONFIRMATION
# ============================================================
generated_files = [
    "nifty_spot_clean.csv",
    "banknifty_spot_clean.csv",
    "nifty_futures_clean.csv",
    "banknifty_futures_clean.csv",
    "nifty_options_clean.csv",
    "banknifty_options_clean.csv",
    "india_vix_clean.csv",
    "risk_free_rate_clean.csv",
]

print("=" * 42)
print()
print("Data Cleaning Completed Successfully")
print()
print("Files Generated:")
print()
for file_name in generated_files:
    exists = (CLEANED_DIR / file_name).exists()
    marker = "\u2713" if exists else "\u2717 (missing)"
    print(f"{marker} {file_name}")
print()
print("=" * 42)



Data Cleaning Completed Successfully

Files Generated:

✓ nifty_spot_clean.csv
✓ banknifty_spot_clean.csv
✓ nifty_futures_clean.csv
✓ banknifty_futures_clean.csv
✓ nifty_options_clean.csv
✓ banknifty_options_clean.csv
✓ india_vix_clean.csv
✓ risk_free_rate_clean.csv

